<a href="https://colab.research.google.com/github/fadiaputriananda/seismology-portfolio/blob/main/seismic_monitoring_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install Libary**

In [1]:
!pip install pandas
!pip install plotly
!pip install gspread
!pip install gspread_dataframe

**Take the USGS Data**

In [2]:
import pandas as pd
from datetime import datetime, timedelta

start_date_str = '2026-01-01'
end_date_str = '2026-05-17'

url = f"https://earthquake.usgs.gov/fdsnws/event/1/query.csv?starttime={start_date_str}&endtime={end_date_str}&minmagnitude=4"
df = pd.read_csv(url)

df.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-05-16T22:30:01.757Z,-2.8870,148.0014,10.000,4.3,mb,12,144,1.051,0.65,...,2026-05-17T09:41:30.040Z,"124 km SE of Lorengau, Papua New Guinea",earthquake,8.37,1.975,0.201,7.0,reviewed,us,us
1,2026-05-16T21:16:17.475Z,13.0317,93.7365,105.224,4.7,mb,68,51,5.935,0.71,...,2026-05-16T21:51:39.040Z,"184 km NE of Bamboo Flat, India",earthquake,8.29,5.923,0.064,75.0,reviewed,us,us
2,2026-05-16T17:59:00.090Z,-22.5537,-70.2765,40.075,4.4,mwr,28,139,0.668,0.70,...,2026-05-16T18:14:32.040Z,"51 km S of Tocopilla, Chile",earthquake,4.97,7.098,0.059,28.0,reviewed,us,us
3,2026-05-16T16:49:18.847Z,36.8493,53.8076,10.000,4.4,mb,89,95,4.389,0.76,...,2026-05-16T19:27:41.040Z,"28 km NE of Behshahr, Iran",earthquake,8.90,1.896,0.062,74.0,reviewed,us,us
4,2026-05-16T15:58:04.288Z,-17.8726,169.1752,10.000,5.4,mww,109,42,3.064,0.58,...,2026-05-18T07:55:03.434Z,"92 km E of Port-Vila, Vanuatu",earthquake,6.66,1.852,0.051,37.0,reviewed,us,us


**Data Cleaning**

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5471 entries, 0 to 5470
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   time             5471 non-null   object 
 1   latitude         5471 non-null   float64
 2   longitude        5471 non-null   float64
 3   depth            5471 non-null   float64
 4   mag              5471 non-null   float64
 5   magType          5471 non-null   object 
 6   nst              5471 non-null   int64  
 7   gap              5471 non-null   int64  
 8   dmin             5471 non-null   float64
 9   rms              5471 non-null   float64
 10  net              5471 non-null   object 
 11  id               5471 non-null   object 
 12  updated          5471 non-null   object 
 13  place            5471 non-null   object 
 14  type             5471 non-null   object 
 15  horizontalError  5461 non-null   float64
 16  depthError       5471 non-null   float64
 17  magError      

In [4]:
#Rename and Choose some columns
df = df.rename(columns={'time': 'Time',
                       'latitude': 'Latitude',
                       'longitude': 'Longitude',
                       'mag': 'Magnitude',
                       'place': 'Place',
                       'depth': 'Depth'})
df = df[['Time','Place','Latitude','Longitude','Magnitude','Depth']]
df.head()

,Time,Place,Latitude,Longitude,Magnitude,Depth
0,2026-05-16T22:30:01.757Z,"124 km SE of Lorengau, Papua New Guinea",-2.8870,148.0014,4.3,10.000
1,2026-05-16T21:16:17.475Z,"184 km NE of Bamboo Flat, India",13.0317,93.7365,4.7,105.224
2,2026-05-16T17:59:00.090Z,"51 km S of Tocopilla, Chile",-22.5537,-70.2765,4.4,40.075
3,2026-05-16T16:49:18.847Z,"28 km NE of Behshahr, Iran",36.8493,53.8076,4.4,10.000
4,2026-05-16T15:58:04.288Z,"92 km E of Port-Vila, Vanuatu",-17.8726,169.1752,5.4,10.000


In [5]:
#Make Country Columns from dataframe
df['Country'] = df['Place'].apply(lambda x: x.split(',')[-1])
df['Country'] = df['Country'].str.strip()
df.head()

,Time,Place,Latitude,Longitude,Magnitude,Depth,Country
0,2026-05-16T22:30:01.757Z,"124 km SE of Lorengau, Papua New Guinea",-2.8870,148.0014,4.3,10.000,Papua New Guinea
1,2026-05-16T21:16:17.475Z,"184 km NE of Bamboo Flat, India",13.0317,93.7365,4.7,105.224,India
2,2026-05-16T17:59:00.090Z,"51 km S of Tocopilla, Chile",-22.5537,-70.2765,4.4,40.075,Chile
3,2026-05-16T16:49:18.847Z,"28 km NE of Behshahr, Iran",36.8493,53.8076,4.4,10.000,Iran
4,2026-05-16T15:58:04.288Z,"92 km E of Port-Vila, Vanuatu",-17.8726,169.1752,5.4,10.000,Vanuatu


In [6]:
df.describe()

,Latitude,Longitude,Magnitude,Depth
count,5471.000000,5471.000000,5471.000000,5471.000000
mean,7.300720,50.683988,4.519419,83.440480
std,29.483373,120.783087,0.394257,135.177116
min,-84.493200,-179.997000,4.000000,1.430000
25%,-15.913000,-68.626050,4.300000,10.000000
50%,2.351200,123.584700,4.400000,35.000000
75%,35.010850,143.097650,4.700000,88.720500
max,86.141400,179.998900,7.500000,683.578000


**Data Analysis and Data Visualization**

In [17]:
#Histogram Magnitude

import plotly.express as px
fig = px.histogram(
    df,
    x = "Magnitude",
    nbins = 18,
    title = "Distribution of Earthquake Magnitude"
)
fig.show()


Based on the distribution of earthquake magnitude, the histogram showcases a right-skewed distribution, which is highly typical for earthquake frequency data. The vast majority of the recorded seismic events are concentrated in the lower magnitude spectrum, specifically peaking between 4.2 and 4.6 magnitudes (classified as Light Earthquakes). As the magnitude increases toward moderate and strong levels, the frequency drops exponentially, leaving only a few rare, high-magnitude events above 7.0 at the far right of the scale.

In [18]:
#Scatter Plot Depth vs Magnitude

import plotly.express as px
fig = px.scatter(
    df,
    x = "Depth",
    y = "Magnitude",
    hover_name = "Place",
    title = "Depth vs Magnitude",
)
fig.show()

Based on the scatter plot of the depth value vs the magnitude value, we can simply say that there is no strong linear correlation between earthquake depth and its magnitude. However, the data shows a high concentration of seismic events at shallow depths (0-150 km) with magnitudes between 4.0 and 5.5. Additionally, a distinct cluster of deep-focus earthquakes is visible around the 500-650 km depth mark.

**Interactive Earthquake Map**

In [ ]:
import plotly.express as px

fig = px.scatter_map(
    df,
    lat = 'Latitude',
    lon = 'Longitude',
    size = 'Depth',
    color = 'Magnitude',
    hover_name = 'Place',
    title = "Global Earthquake Monitoring Map",
    zoom = 1
)
fig.show()

Based on the interactive global earthquake map, it can be concluded that most earthquakes occur in the Pacific Ring of Fire, a zone with a high risk of earthquakes because it is where the world’s major tectonic plates converge, particularly in Indonesia, the Philippines, Japan, Tonga, & Vanuatu.

**Save Dataset**

In [19]:
#Save dataset on Google Spread Sheets in real-time

#Connect Google Colab to Google Spreadsheets
from google.colab import auth
auth.authenticate_user()

In [20]:
#Import Library
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

In [21]:
#Authorized Google Sheets
creds, _ = default()
gc = gspread.authorize(creds)

In [22]:
#Open the Google Sheets (Worksheet File)
spreadsheet = gc.open("Seismic Dashboard Data")
worksheet = spreadsheet.sheet1

In [23]:
#Push the Data Frame to the Google Sheets
worksheet.clear()
set_with_dataframe(worksheet, df)

In [24]:
#save dataset in local device
df.to_csv("earthquake_data.csv", index = False)